### Plan de trabajo

1. Describir los objetivos del estudio.
2. Explorar datos:
    - ¿Es necesario convertir los tipos?
    - ¿Hay valores ausentes o duplicados? Si es así, ¿cómo los caracterizarías?
3.  Lleva a cabo el análisis exploratorio de datos:
    - Estudia la conversión en las diferentes etapas del embudo.
    - ¿El número de eventos por usuario está distribuido equitativamente entre las muestras?
    - ¿Hay usuarios que están presentes en ambas muestras?
    - ¿Cómo se distribuye el número de eventos entre los días?
    - ¿Hay alguna peculiaridad en los datos que hay que tener en cuenta antes de iniciar la prueba A/B?
4. Evaluar los resultados de la prueba A/B:
    - ¿Qué puedes decir sobre los resultados de la prueba A/B?
    - Utiliza una prueba z para comprobar la diferencia estadística entre las proporciones
5. Describe tus conclusiones con respecto a la etapa EDA y los resultados de la prueba A/B

### 1. Describir los objetivos del estudio.
Comprobar si el nuevo embudo de pago (Grupo B) mejora la conversión en el embudo (product_page > product_cart > purchase) con al menos +10% en cada etapa.

In [ ]:
# Importar librerías y cargar archivos
import pandas as pd
import numpy as np
from datetime import datetime, date

In [ ]:


EXPERIMENT_CONFIG = {
    "experiment_name": "recommender_system_test",
    "groups": ("A", "B"),
    "region_target": "EU",
    "allocation_expected": 0.15,   # 15% de nuevos usuarios EU
    "enrollment_window": {
        "start": date(2020, 12, 7),
        "end":   date(2020, 12, 21),
    },
    "events_data_end": date(2021, 1, 1),
    "observation_window_days": 14,
    "funnel_events_order": ["product_page", "product_cart", "purchase"],
    "expected_min_uplift_per_stage": 0.10,   # +10% esperado en cada etapa
    "primary_metrics": [
        "page_rate",                # % usuarios con product_page
        "cart_given_page_rate",     # % con cart dado page
        "purchase_given_cart_rate"  # % con purchase dado cart
    ],
    "secondary_metrics": [
        "page_any", "cart_any", "purchase_any"  # conversión acumulada
    ],
    "stats": {
        "test": "z_proportions",
        "alpha": 0.05,
        "pvalue_adjustment": "holm-bonferroni",  # control por múltiples comparaciones
        "tail": "two-sided"
    },
    "data_quality_checks": [
        "type_conversions", "missing_values", "duplicates",
        "users_in_both_groups", "overlap_with_other_tests",
        "srm_check", "event_volume_balance_by_group",
        "daily_event_distribution"
    ],
    "analysis_policy": {
        # Política de ventanas:
        # - 'strict_full_window': True -> análisis principal solo con 14 días completos por usuario
        # - 'sensitivity_include_truncated': True -> análisis de sensibilidad incluyendo ventanas truncadas
        "strict_full_window": True,
        "sensitivity_include_truncated": True
    }
}

# Impresión de confirmación (no procesa datos aún)
print(">>> Plan del PASO 1 cargado.")
for k, v in EXPERIMENT_CONFIG.items():
    print(f"- {k}: {v}")

In [67]:
# Definir archivos a cargar
path_events = 'final_ab_events_upd_us.csv'
path_users = 'final_ab_new_users_upd_us.csv'
path_parts = 'final_ab_participants_upd_us.csv'
path_mkt = 'ab_project_marketing_events_us.csv'

# Cargar archivos
events = pd.read_csv(path_events)
users = pd.read_csv(path_users)
parts = pd.read_csv(path_parts)
mkt = pd.read_csv(path_mkt)

In [68]:
# Convertir fechas a datetime
events["event_dt"] = pd.to_datetime(events["event_dt"], errors="coerce", utc=True)
users["first_date"] = pd.to_datetime(users["first_date"], errors="coerce").dt.date
mkt["start_dt"]  = pd.to_datetime(mkt["start_dt"], errors="coerce").dt.date
mkt["finish_dt"] = pd.to_datetime(mkt["finish_dt"], errors="coerce").dt.date

# Verificar duplicados (renglones exactos)
def dup_summary(df, name):
    n_dups = df.duplicated().sum()
    print(f"- {name}: duplicados exactos = {n_dups:,} ({n_dups/len(df)*100:.3f}%)")

dup_summary(events, "events")
dup_summary(users, "users")
dup_summary(parts, "parts")
dup_summary(mkt, "mkt")

- events: duplicados exactos = 0 (0.000%)
- users: duplicados exactos = 0 (0.000%)
- parts: duplicados exactos = 0 (0.000%)
- mkt: duplicados exactos = 0 (0.000%)


In [69]:
# Validar rangos de fechas 
def date_range_report():
    print("\n--- Rangos de fecha y hora ---")
    if "event_dt" in events.columns:
        print(f"events.event_dt:   Del {events['event_dt'].min()} al {events['event_dt'].max()}")
    if "first_date" in users.columns:
        print(f"users.first_date:  Del {users['first_date'].min()} al {users['first_date'].max()}")
    if "start_dt" in mkt.columns and "finish_dt" in mkt.columns:
        print(f"mkt.start_dt:      Del {mkt['start_dt'].min()} al {mkt['start_dt'].max()}")
        print(f"mkt.finish_dt:     Del {mkt['finish_dt'].min()} al {mkt['finish_dt'].max()}")

date_range_report()


--- Rangos de fecha y hora ---
events.event_dt:   Del 2020-12-07 00:00:33+00:00 al 2020-12-30 23:36:33+00:00
users.first_date:  Del 2020-12-07 al 2020-12-23
mkt.start_dt:      Del 2020-01-25 al 2020-12-30
mkt.finish_dt:     Del 2020-02-07 al 2021-01-07


In [70]:
# Validar que los usuarios solo pertenezcan a un solo grupo de prueba
# Detectar usuarios en otras pruebas simultáneas

# Filtro al test objetivo
parts_rec = parts[parts["ab_test"] == "recommender_system_test"].copy()
print(f"\nParticipantes en recommender_system_test: {parts_rec['user_id'].nunique():,}")

# Verificar que no existan usuarios en ambos grupos de prueba
overlap_same_test = (
    parts_rec.groupby("user_id")["group"].nunique()
    .reset_index(name="n_groups")
    .query("n_groups > 1")
)
print(f"Usuarios en ambos grupos (mismo test): {overlap_same_test.shape[0]:,}")

# Usuarios del test objetivo que además están en OTROS tests (posible contaminación)
parts_other = parts[parts["ab_test"] != "recommender_system_test"]
users_other_tests = set(parts_other["user_id"]) & set(parts_rec["user_id"])
print(f"Usuarios del test objetivo también en otros tests: {len(users_other_tests):,}")



Participantes en recommender_system_test: 3,675
Usuarios en ambos grupos (mismo test): 0
Usuarios del test objetivo también en otros tests: 887



## Hallazgo importante
Se encontraron 887 usuarios del test objetivo en otros grupos de prueba; lo cual significa que existe contaminación en los datos, por lo que es imprescindible limpiar la cohorte principal, excluyendo a los 887 de ambos grupos para el análisis primario.

Sin embargo, mantendremos el dataframe original para realizar pruebas posteriores para determinar el impacto real de esta situación.

In [71]:
# Filtrar participantes del test objetivo
parts_rec = parts[parts["ab_test"] == "recommender_system_test"].copy()

# Usuarios del test objetivo que también están en otros tests
parts_other = parts[parts["ab_test"] != "recommender_system_test"]
users_other_tests = set(parts_other["user_id"])
overlap_users = set(parts_rec["user_id"]) & users_other_tests

print(f"Usuarios del test objetivo también en otros tests: {len(overlap_users)}")

# Cohorte limpia (sin  repetidos)
parts_rec_clean = parts_rec[~parts_rec["user_id"].isin(overlap_users)].copy()
print(f"Participantes limpios: {parts_rec_clean['user_id'].nunique():,}")

# Cohorte completa (con repetidos)
parts_rec_sens = parts_rec.copy()

# Guardar listas de usuarios por cohorte para usar en el EDA/embudo
u_clean = set(parts_rec_clean["user_id"])
u_sens  = set(parts_rec_sens["user_id"])

print(f"- Usuarios cohorte limpia: {len(u_clean):,}")
print(f"- Usuarios cohorte sensibilidad: {len(u_sens):,}")

Usuarios del test objetivo también en otros tests: 887
Participantes limpios: 2,788
- Usuarios cohorte limpia: 2,788
- Usuarios cohorte sensibilidad: 3,675


In [72]:
# ============================================
# PASO 3.1 · Preparar cohortes y ventanas de observación
# - Cohorte limpia: excluye usuarios que estaban en otros tests
# - Filtrar por región objetivo (EU) y ventana de inscripción
# - Construir ventana de 14 días por usuario (estricta y sensibilidad)
# ============================================
from datetime import timedelta

# 1) Partimos de parts_rec (del paso 2) y de overlap_users (usuarios contaminados)
parts_rec = parts[parts["ab_test"] == EXPERIMENT_CONFIG["experiment_name"]].copy()
parts_other = parts[parts["ab_test"] != EXPERIMENT_CONFIG["experiment_name"]]
overlap_users = set(parts_rec["user_id"]) & set(parts_other["user_id"])

# Cohorte limpia (excluye contaminados)
parts_clean = parts_rec[~parts_rec["user_id"].isin(overlap_users)].copy()

# 2) Filtrar users por región y ventana de inscripción
enroll_start = EXPERIMENT_CONFIG["enrollment_window"]["start"]
enroll_end   = EXPERIMENT_CONFIG["enrollment_window"]["end"]

users_eu = users.query("region == @EXPERIMENT_CONFIG['region_target']").copy()
users_eu_win = users_eu[
    (users_eu["first_date"] >= enroll_start) &
    (users_eu["first_date"] <= enroll_end)
].copy()

# 3) Unir cohortes con usuarios válidos (EU + ventana)
parts_clean_eu = parts_clean.merge(
    users_eu_win[["user_id","first_date","device","region"]],
    on="user_id", how="inner"
)

# 4) Construir ventanas de observación por usuario
#    - Ventana estricta (full 14 días) y sensibilidad (truncada por fin de datos)
events_end_date = EXPERIMENT_CONFIG["events_data_end"]
obs_days = EXPERIMENT_CONFIG["observation_window_days"]

parts_clean_eu["obs_start"] = pd.to_datetime(parts_clean_eu["first_date"])
parts_clean_eu["obs_end_target"] = parts_clean_eu["obs_start"] + pd.to_timedelta(obs_days, unit="D")
parts_clean_eu["obs_end_data"]   = pd.to_datetime(events_end_date)

# Ventana real para sensibilidad (min entre 14 días y fin de datos)
parts_clean_eu["obs_end_real"] = parts_clean_eu[["obs_end_target","obs_end_data"]].min(axis=1)

# Flag de ventana completa de 14 días
parts_clean_eu["has_full_window_14d"] = parts_clean_eu["obs_end_real"].sub(parts_clean_eu["obs_start"]) >= pd.Timedelta(days=obs_days)

print("Cohorte limpia EU (con ventana):", parts_clean_eu.shape)
parts_clean_eu.head(3)


NameError: name 'EXPERIMENT_CONFIG' is not defined